# PEG-ST run analysis

Set `RUN_DIR` to a completed run directory and execute the cells to inspect accuracy, timestep behavior, activity, SOPs, prediction error, and early-exit curves.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RUN_DIR = Path('../runs/dvs128gesture/predictive_aux_T8')

def read_csv(name):
    path = RUN_DIR / name
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

metrics = read_csv('metrics.csv')
timestep = read_csv('timestep_metrics.csv')
stage_activity = read_csv('stage_activity.csv')
sops = read_csv('sops_estimate.csv')
prediction = read_csv('prediction_error.csv')
early_exit = read_csv('early_exit_summary.csv')

display(metrics.tail())
display(timestep.head())
display(stage_activity)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

if not metrics.empty:
    axes[0, 0].plot(metrics['epoch'], metrics['acc1'], label='train')
    if 'val_acc1' in metrics:
        axes[0, 0].plot(metrics['epoch'], metrics['val_acc1'], label='val')
    axes[0, 0].set_title('Accuracy')
    axes[0, 0].legend()

if not timestep.empty:
    axes[0, 1].plot(timestep['timestep'], timestep['accuracy'], marker='o')
    axes[0, 1].set_title('Timestep accuracy')

if not stage_activity.empty:
    axes[1, 0].bar(stage_activity['network_stage'], stage_activity['weighted_output_firing_rate'])
    axes[1, 0].set_title('Stage firing rate')
    axes[1, 0].tick_params(axis='x', rotation=30)

if not prediction.empty and 'normalized_error' in prediction:
    last = prediction.groupby('stage')['normalized_error'].last()
    axes[1, 1].bar(last.index, last.values)
    axes[1, 1].set_title('Prediction error')
    axes[1, 1].tick_params(axis='x', rotation=30)

plt.tight_layout()

In [ ]:
if not early_exit.empty:
    display(early_exit)
    plt.figure(figsize=(6, 4))
    plt.plot(early_exit['avg_exit_timestep'], early_exit['accuracy'], marker='o')
    plt.xlabel('Average exit timestep')
    plt.ylabel('Accuracy')
    plt.title('Early-exit trade-off')
    plt.grid(True)
else:
    print('No early-exit summary found for this run.')